## Notebook for calculating lifetimes & average hydrogen bond lengths

#### System: ACN/water/TBAClO4

**Created on 1st February, 2024**

In [1]:
import math as m
import os, sys
import numpy as np
import argparse
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import find_peaks
import warnings
warnings.filterwarnings('ignore')
from scipy.optimize import curve_fit

import MDAnalysis as mda
# from MDAnalysis.tests.datafiles import TPR, XTC
from MDAnalysis import transformations
from MDAnalysis.analysis.rdf import InterRDF
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA

from dask.distributed import Client
import dask.dataframe as dd
from dask.diagnostics import ProgressBar

In [3]:
# os.chdir('/project/dfreedman/riteshk/COR-MD-hannah/CMD_Rev/')

In [2]:
%%bash
pwd
ls -ltr

/project2/chibueze/riteshk/COR-MD-hannah/CMD_Rev
total 74
-rw-rw-r-- 1 riteshk pi-chibueze 14110 Mar  8 15:33 gen_solv-dist_Hbond_DMF.ipynb
-rw-rw-r-- 1 riteshk pi-chibueze   812 Mar  9 23:45 convert_lammpsdump_xyz.py
drwxrwsr-x 4 riteshk pi-chibueze  4096 Mar 10 13:15 TBAClO4-water_x0.5-solv
drwxrwsr-x 5 riteshk pi-chibueze  4096 Mar 10 21:52 TBAClO4-water_x0.25-solv
-rw-rw-r-- 1 riteshk pi-chibueze   139 Mar 10 22:45 get_cell_dimension.py
drwxrwsr-x 6 riteshk pi-chibueze  4096 Mar 11 08:36 TBAClO4-water_xl1-solv
drwxrwsr-x 5 riteshk pi-chibueze  4096 Mar 11 08:37 TBAClO4-water_xl6-solv
-rw-rw-r-- 1 riteshk pi-chibueze   736 Mar 11 08:41 save_particular_frame_whole.py
-rw-rw-r-- 1 riteshk pi-chibueze   252 Mar 11 08:42 part_traj_save.sh
drwxrwsr-x 6 riteshk pi-chibueze  4096 Mar 11 10:53 TBAClO4-water_xl2-solv
drwxrwsr-x 5 riteshk pi-chibueze  4096 Mar 11 10:55 TBAClO4-water_xl3-solv
drwxrwsr-x 5 riteshk pi-chibueze  4096 Mar 11 10:55 TBAClO4-water_xl4-solv
drwxrwsr-x 4 riteshk pi-chi

In [9]:
client = Client(n_workers=12, threads_per_worker=12)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 12
Total threads: 144,Total memory: 59.53 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39583,Workers: 12
Dashboard: http://127.0.0.1:8787/status,Total threads: 144
Started: Just now,Total memory: 59.53 GiB
Comm: tcp://127.0.0.1:33245,Total threads: 12
Dashboard: http://127.0.0.1:32863/status,Memory: 4.96 GiB
Nanny: tcp://127.0.0.1:34201,


**water mole fractions: l1, l2, l3, l4, l5, l6, 0.25, 0.50**

In [3]:
# import numpy as np
# from scipy.optimize import curve_fit
# from MDAnalysis.analysis.hbonds.hbond_analysis import HydrogenBondAnalysis as HBA

class HydrogenBondCalculator:
    def __init__(self, universe, solv_resname, solv_atom):
        self.u = universe
        self.solv_resname = solv_resname
        self.solv_atom = solv_atom
        self.hbonds_0 = None
        self.hbonds_1 = None

    def fit_biexponential(self, tau_timeseries, ac_timeseries):
        def model(t, A, tau1, B, tau2):
            return A * np.exp(-t / tau1) + B * np.exp(-t / tau2)

        params, params_covariance = curve_fit(model, tau_timeseries, ac_timeseries, [1, 1, 1, 2], maxfev=100000)

        fit_t = np.linspace(tau_timeseries[0], tau_timeseries[-1], 1000)
        fit_ac = model(fit_t, *params)

        return params, fit_t, fit_ac

    def average_num_hbonds(self, hbonds):
        for donor, acceptor, count in hbonds.count_by_type():
            donor_resname, donor_type = donor.split(":")
            n_donors = self.u.select_atoms(f"resname {donor_resname} and type {donor_type}").n_atoms
            mean_count = 2 * int(count) / (hbonds.n_frames * n_donors)
            return mean_count

    def calculate(self):
        self.hbonds_0 = HBA(self.u, 
                            donors_sel=f"resname wat and name O*", 
                            hydrogens_sel=f"resname wat and name H*", 
                            acceptors_sel=f"resname {self.solv_resname} and name {self.solv_atom}",
                            between=["resname wat", f"resname {self.solv_resname}"])
        self.hbonds_0.run()

        tau_max = 25
        window_step = 1

        tau_frames_0, hbond_lifetime_0 = self.hbonds_0.lifetime(tau_max=tau_max, window_step=window_step)

        params_0, fit_t_0, fit_ac_0 = self.fit_biexponential(tau_frames_0, hbond_lifetime_0)
        A, tau1, B, tau2 = params_0
        time_constant_0 = A * tau1 + B * tau2

        num_0 = self.average_num_hbonds(self.hbonds_0)

        self.hbonds_1 = HBA(self.u, 
                            donors_sel="resname wat and name O*", 
                            hydrogens_sel="resname wat and name H*", 
                            acceptors_sel="resname wat and name O*",
                            between=["resname wat", "resname wat"])
        self.hbonds_1.run()

        tau_frames_1, hbond_lifetime_1 = self.hbonds_1.lifetime(tau_max=tau_max, window_step=window_step)

        params_1, fit_t_1, fit_ac_1 = self.fit_biexponential(tau_frames_1, hbond_lifetime_1)

        A, tau1, B, tau2 = params_1
        time_constant_1 = A * tau1 + B * tau2

        num_1 = self.average_num_hbonds(self.hbonds_1)

        return time_constant_0, num_0, time_constant_1, num_1

In [5]:
path_0 = './TBAClO4-water_xl1-solv/ACN/'
path_1 = './TBAClO4-water_xl2-solv/ACN/'
path_2 = './TBAClO4-water_xl3-solv/ACN/'
path_3 = './TBAClO4-water_xl4-solv/ACN/'
path_4 = './TBAClO4-water_xl5-solv/ACN/'
path_5 = './TBAClO4-water_xl6-solv/ACN/'
path_6 = './TBAClO4-water_x0.25-solv/ACN/'
path_7 = './TBAClO4-water_x0.5-solv/ACN/'

In [12]:
u_0 = mda.Universe(path_0 + 'config.pdb', path_0 + 'nvt3.lammpsdump')
u_1 = mda.Universe(path_1 + 'config.pdb', path_1 + 'nvt3.lammpsdump')
u_2 = mda.Universe(path_2 + 'config.pdb', path_2 + 'nvt3.lammpsdump')
u_3 = mda.Universe(path_3 + 'config.pdb', path_3 + 'nvt3.lammpsdump')
u_4 = mda.Universe(path_4 + 'config.pdb', path_4 + 'nvt3.lammpsdump')
# u_5 = mda.Universe(path_5 + 'config.pdb', path_5 + 'nvt3.lammpsdump')
u_6 = mda.Universe(path_6 + 'config.pdb', path_6 + 'nvt3.lammpsdump')
u_7 = mda.Universe(path_7 + 'config.pdb', path_7 + 'nvt3.lammpsdump')

In [13]:
calculator = HydrogenBondCalculator(u_0, 'acn', 'N*')
time_constant_0_ws, num_0_ws, time_constant_0_ww, num_0_ww = calculator.calculate()
print(f"time_constant for l1 mole fraction = {time_constant_0_ws:.2f} ps")
print("Average # of H-bonds b/w water-solv for l1 mole fraction:", num_0_ws)
print(f"time_constant for l1 mole fraction = {time_constant_0_ww:.2f} ps")
print("Average # of H-bonds b/w water-water for l1 mole fraction:", num_0_ww)

time_constant for l1 mole fraction = 0.24 ps
Average # of H-bonds b/w water-solv for l1 mole fraction: 0.4789611744635378
time_constant for l1 mole fraction = 1.18 ps
Average # of H-bonds b/w water-water for l1 mole fraction: 0.933691360439898


In [16]:
calculator = HydrogenBondCalculator(u_1, 'acn', 'N*')
time_constant_1_ws, num_1_ws, time_constant_1_ww, num_1_ww = calculator.calculate()
print(f"time_constant for l2 mole fraction = {time_constant_1_ws:.2f} ps")
print("Average # of H-bonds b/w water-solv for l2 mole fraction:", num_1_ws)
print(f"time_constant for l2 mole fraction = {time_constant_1_ww:.2f} ps")
print("Average # of H-bonds b/w water-water for l2 mole fraction:", num_1_ww)

time_constant for l2 mole fraction = 0.24 ps
Average # of H-bonds b/w water-solv for l2 mole fraction: 0.3725835346058823
time_constant for l2 mole fraction = 0.98 ps
Average # of H-bonds b/w water-water for l2 mole fraction: 1.32746008213228


In [17]:
calculator = HydrogenBondCalculator(u_2, 'acn', 'N*')
time_constant_2_ws, num_2_ws, time_constant_2_ww, num_2_ww = calculator.calculate()
print(f"time_constant for l3 mole fraction = {time_constant_2_ws:.2f} ps")
print("Average # of H-bonds b/w water-solv for l3 mole fraction:", num_2_ws)
print(f"time_constant for l3 mole fraction = {time_constant_2_ww:.2f} ps")
print("Average # of H-bonds b/w water-water for l3 mole fraction:", num_2_ww)

time_constant for l3 mole fraction = 0.25 ps
Average # of H-bonds b/w water-solv for l3 mole fraction: 0.25733984454586567
time_constant for l3 mole fraction = 0.77 ps
Average # of H-bonds b/w water-water for l3 mole fraction: 1.6795620194413505


In [18]:
calculator = HydrogenBondCalculator(u_3, 'acn', 'N*')
time_constant_3_ws, num_3_ws, time_constant_3_ww, num_3_ww = calculator.calculate()
print(f"time_constant for 0.25 mole fraction = {time_constant_3_ws:.2f} ps")
print("Average # of H-bonds b/w water-solv for l4 mole fraction:", num_3_ws)
print(f"time_constant for 0.25 mole fraction = {time_constant_3_ww:.2f} ps")
print("Average # of H-bonds b/w water-water for l4 mole fraction:", num_3_ww)

time_constant for 0.25 mole fraction = 0.24 ps
Average # of H-bonds b/w water-solv for l4 mole fraction: 0.1835782496838617
time_constant for 0.25 mole fraction = 0.66 ps
Average # of H-bonds b/w water-water for l4 mole fraction: 1.9215087515373033


In [19]:
calculator = HydrogenBondCalculator(u_4, 'acn', 'N*')
time_constant_4_ws, num_4_ws, time_constant_4_ww, num_4_ww = calculator.calculate()
print(f"time_constant for 0.5 mole fraction = {time_constant_4_ws:.2f} ps")
print("Average # of H-bonds b/w water-solv for l5 mole fraction:", num_4_ws)
print(f"time_constant for 0.5 mole fraction = {time_constant_4_ww:.2f} ps")
print("Average # of H-bonds b/w water-water for l5 mole fraction:", num_4_ww)

time_constant for 0.5 mole fraction = 0.24 ps
Average # of H-bonds b/w water-solv for l5 mole fraction: 0.1392005812730216
time_constant for 0.5 mole fraction = 0.59 ps
Average # of H-bonds b/w water-water for l5 mole fraction: 2.0742551036725123


In [10]:
calculator = HydrogenBondCalculator(u_5, 'acn', 'N*')
time_constant_5_ws, num_5_ws, time_constant_5_ww, num_5_ww = calculator.calculate()
print(f"time_constant for l6 mole fraction = {time_constant_5_ws:.2f} ps")
print("Average # of H-bonds b/w water-solv for l6 mole fraction:", num_5_ws)
print(f"time_constant for l6 mole fraction = {time_constant_5_ww:.2f} ps")
print("Average # of H-bonds b/w water-water for l6 mole fraction:", num_5_ww)

/scratch/midway3/riteshk/anaconda3/lib/python3.9/site-packages/MDAnalysis/analysis/hydrogenbonds/hbond_analysis.py:672: DeprecationWarning: The `hbonds` attribute was deprecated in MDAnalysis 2.0.0 and will be removed in MDAnalysis 3.0.0. Please use `results.hbonds` instead.
  warnings.warn(wmsg, DeprecationWarning)


time_constant for l6 mole fraction = 0.24 ps
Average # of H-bonds b/w water-solv for l6 mole fraction: 0.11647742333581022
time_constant for l6 mole fraction = 0.55 ps
Average # of H-bonds b/w water-water for l6 mole fraction: 2.1514904813429108


In [14]:
calculator = HydrogenBondCalculator(u_6, 'acn', 'N*')
time_constant_6_ws, num_6_ws, time_constant_6_ww, num_6_ww = calculator.calculate()
print(f"time_constant for 0.25 mole fraction = {time_constant_6_ws:.2f} ps")
print("Average # of H-bonds b/w water-solv for 0.25 mole fraction:", num_6_ws)
print(f"time_constant for 0.25 mole fraction = {time_constant_6_ww:.2f} ps")
print("Average # of H-bonds b/w water-water for 0.25 mole fraction:", num_6_ww)

time_constant for 0.25 mole fraction = 0.25 ps
Average # of H-bonds b/w water-solv for 0.25 mole fraction: 0.15405838920834083
time_constant for 0.25 mole fraction = 0.61 ps
Average # of H-bonds b/w water-water for 0.25 mole fraction: 2.018483187241652


In [15]:
calculator = HydrogenBondCalculator(u_7, 'acn', 'N*')
time_constant_7_ws, num_7_ws, time_constant_7_ww, num_7_ww = calculator.calculate()
print(f"time_constant for 0.5 mole fraction = {time_constant_7_ws:.2f} ps")
print("Average # of H-bonds b/w water-solv for 0.5 mole fraction:", num_7_ws)
print(f"time_constant for 0.5 mole fraction = {time_constant_7_ww:.2f} ps")
print("Average # of H-bonds b/w water-water for 0.5 mole fraction:", num_7_ww)

distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)


time_constant for 0.5 mole fraction = 0.24 ps
Average # of H-bonds b/w water-solv for 0.5 mole fraction: 0.0885584774833146
time_constant for 0.5 mole fraction = 0.51 ps
Average # of H-bonds b/w water-water for 0.5 mole fraction: 2.251237162584605


In [22]:
mol_frac_wat = ['l1', 'l2', 'l3', 'l4', 'l5', 'l6', 0.25, 0.50]
time_const_ws_arr = [time_constant_0_ws, time_constant_1_ws, time_constant_2_ws, time_constant_3_ws, time_constant_4_ws, time_constant_5_ws, time_constant_6_ws, time_constant_7_ws]
time_const_ww_arr = [time_constant_0_ww, time_constant_1_ww, time_constant_2_ww, time_constant_3_ww, time_constant_4_ww, time_constant_5_ww, time_constant_6_ww, time_constant_7_ww]
num_ws_arr = [num_0_ws, num_1_ws, num_2_ws, num_3_ws, num_4_ws, num_5_ws, num_6_ws, num_7_ws]
num_ww_arr = [num_0_ww, num_1_ww, num_2_ww, num_3_ww, num_4_ww, num_5_ww, num_6_ww, num_7_ww]
dict_fin = {'water_mol_frac': mol_frac_wat, 'lifetime_water-solv': time_const_ws_arr, 'lifetime_water-water': time_const_ww_arr, 'avg_num_hb_water-solv': num_ws_arr, 'avg_num_hb_water-water': num_ww_arr}
df_fin = pd.DataFrame(dict_fin)
df_fin

,water_mol_frac,lifetime_water-solv,lifetime_water-water,avg_num_hb_water-solv,avg_num_hb_water-water
0,l1,0.238746,1.177019,0.478961,0.933691
1,l2,0.242503,0.984345,0.372584,1.327460
2,l3,0.248603,0.767880,0.257340,1.679562
3,l4,0.241526,0.657180,0.183578,1.921509
4,l5,0.239456,0.588516,0.139201,2.074255
5,l6,0.243944,0.554910,0.116477,2.151490
6,0.25,0.249540,0.611829,0.154058,2.018483
7,0.5,0.239410,0.514368,0.088558,2.251237


In [23]:
df_fin.to_csv('data_lifetime_num_hb_acn.csv', index=False)